# Induced Surface Charge

This example follows charge moving from a conductor’s interior towards its surface. An external electric field also separates positive and negative charge across the body. The resulting distribution depends on its shape and on the charge it started with.

The electric displacement and the conduction current respond to the same spacetime field. Displacement determines the charge present; current determines how that charge changes. Their material maps differ by permittivity and conductivity.

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"
%matplotlib inline

import numpy as np
from functools import reduce
from operator import and_
from numga import Algebra, NumpyContext, stack, where
from numga.sparse import spdiag
from examples.util.animation import save_figure, show_states
from examples.electromagnetism.induced_charge import render, scenarios

ga = Algebra("x+y+t-w0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Field = ga.gatype.bivector()                        # Bivector+
origin = mv.w.dual()                               # [] Point-
time = -mv.t.dual()                                # [] Point-
axes = stack((mv.x.dual(), mv.y.dual()))             # [axes] Point-
sweeps = axes & time                               # [axes] Excitation-


## Isolated conductors

A negatively charged ring and a positively charged rectangular rod sit in an insulator. Both have finite conductivity, falling to zero across a thin material transition. The surrounding boundary supplies the applied field, while the insulating gaps prevent charge transfer between the bodies or to that boundary.

In [ ]:
n_across, n_frames = scenarios.n_across, scenarios.n_frames
permittivity, conductivities = scenarios.permittivity, scenarios.conductivities
body_charges = scenarios.body_charges
frame_step, n_substeps = scenarios.frame_step, scenarios.n_substeps
frame_duration_ms = scenarios.frame_duration_ms
topology, vertices, fractions, memberships, boundary = scenarios.scene(n_across, n_frames)
metal = fractions.sum(axis=0)
save_figure(render.setup(topology, vertices, metal), "charge_relaxation_notebook_setup");


## Electric storage and conduction

An electric field reads the patches swept through time by spatial displacements. The response below returns an intrinsic excitation that pairs with the field. Multiplying `field & electric_response(field)` by permittivity gives twice the stored electric energy density; multiplying by conductivity gives the dissipated power density.

In [ ]:
electric_response = (sweeps * (Field & sweeps)).sum(axis=0)
corners = topology.incidence[2, 0]
mean = corners.mean(axis=0)
spans = reduce(and_, corners * vertices, time)
areas = spans.norm() / 2                           # Scalar+[faces]
cells = ~mean * areas                              # Scalar+[vertices]
facing = topology.off[2, 0] * vertices
complements = facing[0] & facing[1] & time
hats = complements / (complements & (corners * vertices))
gradients = hats - (hats & origin) * mv.w           # [corners] Plane+[faces]
hat_fields = -(gradients ^ mv.t)                   # [corners] Field+[faces]

# A potential change contributes to both electric storage and conduction through this pairing.
local = areas * (hat_fields[:, None] & electric_response(hat_fields[None, :]))
conductivity = (conductivities * (mean * fractions)).sum(axis=0)
charge_from_voltage = (~corners[:, None] * spdiag(permittivity * local)
                       * corners[None, :]).sum(axis=0).sum(axis=0)
outflow_from_voltage = (~corners[:, None] * spdiag(conductivity * local)
                        * corners[None, :]).sum(axis=0).sum(axis=0)


## Charge in the interior

Each body's initial excess is spread through its bulk. The potential responds to both charge distributions and the applied field, including inside the conductors.

Charge is the outward balance of electric displacement around each vertex's cell. A rapid change in its outward component near a material edge therefore appears as concentrated charge. Conductivity determines where charge can move; its accumulation at a material edge emerges from that motion.

In [ ]:
extension = ~topology.reading(topology.interior[0], 0)
poisson = (~extension * charge_from_voltage * extension) * Scalar
parcels = cells * fractions
initial_charge = (body_charges * parcels / parcels.sites.sum()).sum(axis=0)


## Charge relaxation

Conduction redistributes charge until the electric field inside each body weakens. The field produced by the redistributed charge opposes the field that drove the motion. Turning the applied field disturbs that balance and starts the redistribution again.

The applied field is initially held still, turns through a half-revolution, then remains still while charge settles. This electroquasistatic model follows charge relaxation while neglecting magnetic induction and electromagnetic propagation. Computing and drawing the animation takes several seconds.

In [ ]:
def relaxation():
    """The evolving potential and integrated charge occupy the same mesh vertices."""
    charge = initial_charge
    held = boundary[0]
    potential = held + extension * poisson.solve(~extension * (charge - charge_from_voltage * held))
    yield potential, charge

    step = frame_step / n_substeps
    for target in boundary[1:]:
        previous = held
        for substep in range(n_substeps):
            # Equal and opposite transfers conserve each isolated body's initial charge.
            charge = charge - step * (outflow_from_voltage * potential)
            held = previous + ((substep + 1) / n_substeps) * (target - previous)
            potential = held + extension * poisson.solve(~extension * (charge - charge_from_voltage * held))
        yield potential, charge


states = show_states(relaxation(), render.animate, topology, vertices, metal,
                     name="opposite_charge_potential_notebook", duration_ms=frame_duration_ms)


Red and blue show positive and negative charge density; higher densities become lighter on a fixed colour scale. Gold curves are equipotentials, and ice-blue streaks follow the electric field across them. Longer streaks indicate a stronger field, and the texture fades where the field weakens, using fixed scales throughout the animation. Both include the bodies' charges and the applied field. Colour gathers towards the surfaces as charge leaves the interiors.

## Conserved excess charge

Each isolated conductor keeps its initial total charge while the distribution responds to the changing applied field. The ring remains negatively charged and the rod positively charged. Body membership is used only to read these totals.

In [ ]:
charges = stack([state[1] for state in states])
net = (charges[:, None] * memberships).sites.sum()
negative = (where(charges < 0, -charges, 0)[:, None] * memberships).sites.sum()
save_figure(render.balance(net, negative, frame_step), "opposite_charge_relaxation_notebook_balance");


Dashed curves show the conserved total charge, blue for the ring and orange for the rod. Solid curves show the magnitude of negative charge on each. Their changing heights measure redistribution while the dashed totals stay fixed.

The [COMSOL electroquasistatics overview](https://www.comsol.com/multiphysics/electroquasistatics-theory) describes the charge-conservation and displacement-current approximation used here.

## Charge near the corners

The final charge density can be read along the rod to distinguish concentration at its corners from the broad polarization of its sides. The curve below gives the largest absolute density across the material layer at each position along the rod; dotted lines mark its ends.

In [ ]:
save_figure(render.corner_density(topology, vertices, charges[-1], scenarios.rod_centre,
                                  scenarios.rod_turn, scenarios.rod_lengths, scenarios.interface_width),
            "rod_corner_charge_density_notebook");


In [ ]:
# checks
# Both isolated bodies keep the charge they started with.
assert np.all((net - body_charges).abs() < .01)
